In [ ]:
import logging
import sys
import torch.distributed as dist
from collections.abc import MutableMapping
from logging import getLogger

from ray import tune

from recbole.config import Config
from recbole.data import (
    create_dataset,
    data_preparation,
)
from recbole.data.transform import construct_transform
from recbole.utils import (
    init_logger,
    get_model,
    get_trainer,
    init_seed,
    set_color,
    get_flops,
    get_environment,
)
import torch
import os
from recbole.data.interaction import Interaction




In [ ]:
dataset_name = "Amazon_Instruments"
if dataset_name == 'Amazon_Arts':
    model_path = '/root/autodl-tmp/SRMR-Rec/saved/DMF-Amazon_Arts-Jul-30-2026_16-46-17.pth'
elif dataset_name == "Amazon_Games":
    model_path = '/root/autodl-tmp/SRMR-Rec/saved/DMF-Amazon_Games-Jul-30-2026_15-53-34.pth'
elif dataset_name == "Amazon_Instruments":
    model_path = '/root/autodl-tmp/SRMR-Rec/saved/DMF-Amazon_Instruments-Jul-30-2026_15-57-20.pth'
data_path = f'/root/autodl-tmp/SRMR-Rec/datasets/{dataset_name}'




In [ ]:

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config, model, dataset, train_data, valid_data, test_data  = load_data_and_model(
    model_file=model_path,
    data_path=data_path
)
model



In [ ]:
user_id = torch.tensor([1])
user_embedding = model.get_user_embedding(user=user_id)
user_embedding

In [ ]:
item_embeddings = model.get_item_embedding()

In [ ]:
dataset.field2token_id['item_id']

In [ ]:
item_embeddings[dataset.field2token_id['item_id']['B0002D0NJ8']]

In [ ]:
from recbole.data import interaction
dic_inter = {'user_id' : torch.tensor([1,]), "item_id" : torch.tensor([10,])}
new_inter = interaction.Interaction(dic_inter)
model.predict(new_inter)

In [ ]:
import numpy as np
arr = model.get_user_embedding(user=user_id).cpu().detach().numpy().reshape(-1)
arr

In [ ]:
len(dataset.field2id_token['user_id'])

In [ ]:
dataset.dataset_name

In [ ]:
# from sklearn.cluster import KMeans
# # load user_id from the dataset
# user_id = dataset.field2id_token['user_id']

# # cluster user_id from model's user embeddings
# uid_range = len(user_id)

# total_user_emb = []

# # ignore padding user
# for i in range(1, uid_range):
#     user_emb = model.get_user_embedding(torch.tensor([i, ])).cpu().detach().numpy().reshape(-1)
#     total_user_emb.append(user_emb)

# # use KMeans to cluster user embeddings
# n_clusters = 100
# random_seed = 2024
# kmeans = KMeans(n_clusters=n_clusters, random_state=random_seed).fit(total_user_emb)
# user_cluster = kmeans.labels_

In [ ]:
# user_cluster = user_cluster.tolist()

In [ ]:
# user_cluster

In [ ]:
# use leave_one_out to get the dataset
datasets = dataset.leave_one_out(group_by='user_id', leave_one_mode='valid_and_test')
train_dataset, valid_dataset, test_dataset = datasets[0], datasets[1], datasets[2]

In [ ]:
# get user's history interactions, rating and length (train_dataset as example)
train_history_matrix, train_history_value, train_length = train_dataset.history_item_matrix('timestamp')
valid_history_matrix, valid_history_value, valid_length = valid_dataset.history_item_matrix('timestamp')
test_history_matrix, test_history_value, test_length = test_dataset.history_item_matrix('timestamp')

In [ ]:
for i in range(train_history_matrix.shape[0]):
    arr = train_history_value[i]
    # sort indices with reversed sequence
    sorted_arr = torch.argsort(arr, descending=False)
    train_history_matrix[i] = train_history_matrix[i][sorted_arr]
    train_history_value[i] = train_history_value[i][sorted_arr]
    # move zero elements to the end
    zero_idx = torch.where(train_history_matrix[i] == 0)[0]
    non_zero_idx = torch.where(train_history_matrix[i] != 0)[0]
    train_history_matrix[i] = torch.cat([train_history_matrix[i][non_zero_idx], train_history_matrix[i][zero_idx]])
    train_history_value[i] = torch.cat([train_history_value[i][non_zero_idx], train_history_value[i][zero_idx]])

In [ ]:
# get user's history interactions
train_history_matrix[1]

In [ ]:
train_history_value[1]

In [ ]:
# get user's history interaction length
# history_len[15403]

In [ ]:
len(dataset.field2token_id['item_id'])

In [ ]:
len(train_dataset.field2token_id['item_id'])

In [ ]:
len(dataset.field2token_id['user_id'])

In [ ]:
len(train_dataset.field2token_id['user_id'])

## get the interaction sequence of every user

In [ ]:
# concat train, valid, test history matrix 

In [ ]:
# train
train_item_re_id_inters_dict = {}
for uid_int in range(1, train_history_matrix.size(0)):
    train_item_re_id_inters_dict[uid_int] = train_history_matrix[uid_int].tolist()[:train_length[uid_int]]
train_item_re_id_inters_dict # user_id -> item_re_id

In [ ]:
# valid
valid_item_re_id_inters_dict = {}
for uid_int in range(1, valid_history_matrix.size(0)):
    valid_item_re_id_inters_dict[uid_int] = valid_history_matrix[uid_int].tolist()[:valid_length[uid_int]]
# test
test_item_re_id_inters_dict = {}
for uid_int in range(1, test_history_matrix.size(0)):
    test_item_re_id_inters_dict[uid_int] = test_history_matrix[uid_int].tolist()[:test_length[uid_int]]

In [ ]:
# concat train, valid, test
item_re_id_inters_dict = {}
for uid_int in train_item_re_id_inters_dict.keys():
    item_re_id_inters_dict[uid_int] = train_item_re_id_inters_dict[uid_int] + valid_item_re_id_inters_dict[uid_int] + test_item_re_id_inters_dict[uid_int]
item_re_id_inters_dict

In [ ]:

item_id_re2ori = {}
for key, value in dataset.field2token_id['item_id'].items():
    item_id_re2ori[value] = key
item_id_re2ori

In [ ]:
# we need the form: user_id -> item_ori_id
item_ori_id_inters_dict = {}
for key, value in item_re_id_inters_dict.items():
    item_ori_id_inters_dict[key] = [item_id_re2ori[item_re_id] for item_re_id in value]
item_ori_id_inters_dict

In [ ]:
dataset.dataset_name

In [ ]:
import json

In [ ]:
with open(f'/root/autodl-tmp/SRMR-Rec/prepare_cluster_user/datasets/{dataset.dataset_name}/new_inters_dict.json','w') as f:
    json.dump(item_ori_id_inters_dict,f)

## data preprocessing

train

In [ ]:
item_ori_id_set = set(dataset.field2token_id['item_id'].keys()) - set(['[PAD]'])
item_ori_id_list = list(item_ori_id_set)

In [ ]:
from tqdm import tqdm
import random
random.seed(2024)

In [ ]:
train_item_ori_id_inters_dict = {}
for uid in tqdm(item_ori_id_inters_dict.keys()):
    items = item_ori_id_inters_dict[uid][:-2]
    train_item_ori_id_inters_dict[uid] = {}
    for i in range(2, len(items)):
        #negative_item_set = item_ori_id_set - set(items[i])
        #candidate_set = [items[i]] + random.sample(list(negative_item_set), 49)
        candidate_set = [items[i]] + random.sample(item_ori_id_list, 49)
        random.shuffle(candidate_set)
        

        train_item_ori_id_inters_dict[uid][i] = {'inters' : items[:i], 
                                                      'target_item' : items[i], 
                                                      'candidate_set' : candidate_set}

In [ ]:
with open(f'/root/autodl-tmp/SRMR-Rec/prepare_cluster_user/datasets/{dataset.dataset_name}/new_train_samples.json','w')as f:
    json.dump(train_item_ori_id_inters_dict,f)

valid

In [ ]:

valid_item_ori_id_inters_dict = {}
item_ori_id_list = list(item_ori_id_set)
for uid in tqdm(item_ori_id_inters_dict.keys()):
    items = item_ori_id_inters_dict[uid][:-1]
    valid_item_ori_id_inters_dict[uid] = {}
    i = -1
    #negative_item_set = item_ori_id_set - set(items[i])
    #candidate_set = [items[i]] + random.sample(list(negative_item_set), 49)
    candidate_set = [items[i]] + random.sample(item_ori_id_list, 49)
    random.shuffle(candidate_set)
    
    valid_item_ori_id_inters_dict[uid][i] = {'inters' : items[:i], 
                                                    'target_item' : items[i], 
                                                    'candidate_set' : candidate_set}

In [ ]:
with open(f'/root/autodl-tmp/SRMR-Rec/prepare_cluster_user/datasets/{dataset.dataset_name}/new_valid_samples.json','w')as f:
    json.dump(valid_item_ori_id_inters_dict,f)

test

In [ ]:
test_item_ori_id_inters_dict = {}
item_ori_id_list = list(item_ori_id_set)
for uid in tqdm(item_ori_id_inters_dict.keys()):
    items = item_ori_id_inters_dict[uid]
    test_item_ori_id_inters_dict[uid] = {}
    i = -1
    #negative_item_set = item_ori_id_set - set(items[i])
    #candidate_set = [items[i]] + random.sample(list(negative_item_set), 49)
    candidate_set = [items[i]] + random.sample(item_ori_id_list, 49)
    random.shuffle(candidate_set)

    test_item_ori_id_inters_dict[uid][i] = {'inters' : items[:i], 
                                                    'target_item' : items[i], 
                                                    'candidate_set' : candidate_set}

In [ ]:
with open(f'/root/autodl-tmp/SRMR-Rec/prepare_cluster_user/datasets/{dataset.dataset_name}/new_test_samples.json','w')as f:
    json.dump(test_item_ori_id_inters_dict,f)